<a href="https://colab.research.google.com/github/shiyamsundarbl-ui/dl-business-analytics/blob/main/notebooks/ch02_tools_math.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #1F3864 0%, #2E5FA3 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #a8c4e8; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #c8ddf5; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.12); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: #ddeeff; font-size: 1.05em; font-weight: 600;">&#128214; Chapter 2 &nbsp;&middot;&nbsp; Essential Tools and Math</span>
  </div>
</div>
<div style="background: #f0f4fa; border-left: 5px solid #2E5FA3; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>NumPy, pandas, and PyTorch tensors — the three tools every deep learning practitioner uses daily, and how they connect to each other.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| 2.1 NumPy Essentials | Arrays, shapes, indexing, vectorised operations, broadcasting |
| 2.2 pandas for Data Work | Series, DataFrame, loading data, cleaning, grouping, filtering |
| 2.3 PyTorch Tensors | Creating tensors, shapes, operations, device placement |
| 2.4 Connecting the Three | Converting between NumPy, pandas, and PyTorch; memory sharing vs copy |
| 2.5 Linear Algebra Essentials | Dot product, matrix multiplication, shapes and broadcasting |
| 2.6 Data Pipelines | Dataset class, DataLoader, batching, shuffling |
| 2.7 Normalisation & Preprocessing | Why it matters, StandardScaler, MinMaxScaler, encoding categoricals, train/val/test split |

**Dataset:** UCI Adult Income — predict whether a person earns more than $50K/year based on census features.  
Loaded directly from a URL — no Kaggle account or file download required.

> **This chapter is a toolkit chapter.** Every concept introduced here will be used in every chapter that follows. Take your time, run each cell, and make sure you understand the output before moving on.

---
## Setup

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Chapter 2 — Setup
# Installs and imports everything needed for this chapter.
# Expected time: 1–2 minutes on first run.
# ─────────────────────────────────────────────────────────────────────────────

!pip install --quiet numpy pandas torch matplotlib

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Reproducibility — fix random seeds so your results match the book
np.random.seed(42)
torch.manual_seed(42)

print(f"NumPy  version : {np.__version__}")
print(f"pandas version : {pd.__version__}")
print(f"PyTorch version: {torch.__version__}")
print()
print("Setup complete. Ready to begin Chapter 2.")

NumPy  version : 2.0.2
pandas version : 2.2.2
PyTorch version: 2.10.0+cpu

Setup complete. Ready to begin Chapter 2.


---
# 2.1 NumPy Essentials

**NumPy** (Numerical Python) is the foundation of scientific computing in Python. Every data science and deep learning library — pandas, PyTorch, scikit-learn — is built on top of NumPy or mirrors its conventions.

The core object is the **ndarray** (n-dimensional array): a grid of numbers, all of the same data type, stored in a contiguous block of memory. This makes NumPy operations orders of magnitude faster than equivalent Python loops.

**Table 2.1 — NumPy vs Python list: key differences**

| | Python list | NumPy array |
|--|------------|-------------|
| Element types | Mixed (int, str, float, ...) | Single type (all float32, all int64, ...) |
| Memory | Scattered (pointers) | Contiguous block |
| Arithmetic | Requires loops | Vectorised — operates on all elements at once |
| Speed | Slow for maths | Very fast |
| Multi-dimensional | Nested lists only | Native n-dimensional support |

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1a — Creating arrays and understanding shape
# shape is the most important property of any array/tensor you will work with.
# It tells you: how many dimensions, and how many elements in each dimension.
# ─────────────────────────────────────────────────────────────────────────────

# 1-D array: a row of numbers (like a single customer's feature vector)
customer_features = np.array([35, 52000, 8, 1])  # age, income, years_exp, has_degree
print("1-D array (one customer):")
print(f"  Values : {customer_features}")
print(f"  Shape  : {customer_features.shape}   ← (4,) means 4 elements, 1 dimension")
print(f"  dtype  : {customer_features.dtype}")
print()

# 2-D array: rows = customers, columns = features (like a mini-dataset)
customer_batch = np.array([
    [35, 52000, 8, 1],   # customer 1
    [28, 34000, 3, 0],   # customer 2
    [45, 78000, 15, 1],  # customer 3
])
print("2-D array (three customers, four features each):")
print(customer_batch)
print(f"  Shape  : {customer_batch.shape}   ← (3, 4) means 3 rows, 4 columns")
print(f"  ndim   : {customer_batch.ndim}     ← number of dimensions")
print(f"  size   : {customer_batch.size}    ← total number of elements (3 × 4)")

1-D array (one customer):
  Values : [   35 52000     8     1]
  Shape  : (4,)   ← (4,) means 4 elements, 1 dimension
  dtype  : int64

2-D array (three customers, four features each):
[[   35 52000     8     1]
 [   28 34000     3     0]
 [   45 78000    15     1]]
  Shape  : (3, 4)   ← (3, 4) means 3 rows, 4 columns
  ndim   : 2     ← number of dimensions
  size   : 12    ← total number of elements (3 × 4)


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1b — Useful array creation functions
# These appear constantly in deep learning code.
# ─────────────────────────────────────────────────────────────────────────────

print("np.zeros — all zeros (common for initialising weight matrices):")
print(np.zeros((3, 4)))
print()

print("np.ones — all ones:")
print(np.ones((2, 3)))
print()

print("np.random.randn — random values from standard normal distribution:")
print(np.random.randn(2, 4).round(4))
print()

print("np.arange — evenly spaced values (like Python range but returns array):")
print(np.arange(0, 1.0, 0.2))   # start, stop, step
print()

print("np.linspace — N evenly spaced values between start and end:")
print(np.linspace(0, 1, 6))     # start, end, count

np.zeros — all zeros (common for initialising weight matrices):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

np.ones — all ones:
[[1. 1. 1.]
 [1. 1. 1.]]

np.random.randn — random values from standard normal distribution:
[[ 0.4967 -0.1383  0.6477  1.523 ]
 [-0.2342 -0.2341  1.5792  0.7674]]

np.arange — evenly spaced values (like Python range but returns array):
[0.  0.2 0.4 0.6 0.8]

np.linspace — N evenly spaced values between start and end:
[0.  0.2 0.4 0.6 0.8 1. ]


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1c — Indexing and slicing
# Selecting rows, columns, and elements from arrays.
# ─────────────────────────────────────────────────────────────────────────────

data = np.array([
    [35, 52000, 8, 1],
    [28, 34000, 3, 0],
    [45, 78000, 15, 1],
    [52, 95000, 22, 1],
])

print("Full array — shape", data.shape)
print(data)
print()

# Row selection
print("First row  (data[0])       :", data[0])
print("Last row   (data[-1])      :", data[-1])
print("Rows 1–2   (data[1:3])     :")
print(data[1:3])
print()

# Column selection
print("All incomes (column 1, data[:, 1]) :", data[:, 1])
print("First two columns (data[:, :2])    :")
print(data[:, :2])
print()

# Boolean indexing — select rows where income > 50000
high_income_mask = data[:, 1] > 50000
print("Boolean mask (income > 50k):", high_income_mask)
print("High-income customers only :")
print(data[high_income_mask])

Full array — shape (4, 4)
[[   35 52000     8     1]
 [   28 34000     3     0]
 [   45 78000    15     1]
 [   52 95000    22     1]]

First row  (data[0])       : [   35 52000     8     1]
Last row   (data[-1])      : [   52 95000    22     1]
Rows 1–2   (data[1:3])     :
[[   28 34000     3     0]
 [   45 78000    15     1]]

All incomes (column 1, data[:, 1]) : [52000 34000 78000 95000]
First two columns (data[:, :2])    :
[[   35 52000]
 [   28 34000]
 [   45 78000]
 [   52 95000]]

Boolean mask (income > 50k): [ True False  True  True]
High-income customers only :
[[   35 52000     8     1]
 [   45 78000    15     1]
 [   52 95000    22     1]]


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1d — Vectorised operations and broadcasting
#
# Vectorised: apply the same operation to every element without a loop.
# Broadcasting: NumPy automatically expands arrays of different shapes
#               to make arithmetic work — without copying data.
# ─────────────────────────────────────────────────────────────────────────────

scores = np.array([72, 85, 63, 91, 58])

print("Original scores          :", scores)
print("Add 5 to all (scores + 5):", scores + 5)      # no loop needed
print("Divide all by 100        :", scores / 100)
print("Squared                  :", scores ** 2)
print("Pass/fail (>= 70)        :", scores >= 70)    # vectorised comparison
print()

# Broadcasting: adding a 1-D array to each row of a 2-D array
# This is exactly how biases are added to layer outputs in neural networks
layer_output  = np.array([[1.0, 2.0, 3.0],   # neuron outputs for sample 1
                           [4.0, 5.0, 6.0]])  # neuron outputs for sample 2
bias          = np.array([0.1, 0.2, 0.3])    # one bias per neuron

result = layer_output + bias   # bias broadcasts across both rows automatically
print("layer_output shape:", layer_output.shape,  "  (2 samples, 3 neurons)")
print("bias shape        :", bias.shape,           "  (3 neurons)")
print("After adding bias :")
print(result)
print("\nNo loop was needed. NumPy broadcast bias across both rows automatically.")

Original scores          : [72 85 63 91 58]
Add 5 to all (scores + 5): [77 90 68 96 63]
Divide all by 100        : [0.72 0.85 0.63 0.91 0.58]
Squared                  : [5184 7225 3969 8281 3364]
Pass/fail (>= 70)        : [ True  True False  True False]

layer_output shape: (2, 3)   (2 samples, 3 neurons)
bias shape        : (3,)   (3 neurons)
After adding bias :
[[1.1 2.2 3.3]
 [4.1 5.2 6.3]]

No loop was needed. NumPy broadcast bias across both rows automatically.


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1e — Aggregate functions and reshape
# ─────────────────────────────────────────────────────────────────────────────

data = np.array([
    [35, 52000, 8],
    [28, 34000, 3],
    [45, 78000, 15],
    [52, 95000, 22],
])

print("Array shape:", data.shape, "  (4 customers, 3 features)")
print()

# axis=0 means collapse rows (compute stat per column)
# axis=1 means collapse columns (compute stat per row)
print("Column means (axis=0) — average value per feature:")
print("  ", data.mean(axis=0))

print("Column std   (axis=0) — spread per feature:")
print("  ", data.std(axis=0).round(2))

print("Row sums     (axis=1) — total per customer:")
print("  ", data.sum(axis=1))
print()

# reshape: rearrange elements into a new shape — total elements must stay the same
flat = np.arange(12)          # 12 elements, shape (12,)
matrix = flat.reshape(3, 4)   # same 12 elements, shape (3, 4)
print("flat shape   :", flat.shape)
print("matrix shape :", matrix.shape)
print(matrix)
print()
print("-1 in reshape means: infer this dimension automatically")
print("flat.reshape(-1, 4):", flat.reshape(-1, 4).shape,  "  (12/4 = 3 rows inferred)")
print("flat.reshape(2, -1):", flat.reshape(2, -1).shape,  "  (12/2 = 6 cols inferred)")

Array shape: (4, 3)   (4 customers, 3 features)

Column means (axis=0) — average value per feature:
   [4.000e+01 6.475e+04 1.200e+01]
Column std   (axis=0) — spread per feature:
   [9.190000e+00 2.344542e+04 7.180000e+00]
Row sums     (axis=1) — total per customer:
   [52043 34031 78060 95074]

flat shape   : (12,)
matrix shape : (3, 4)
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

-1 in reshape means: infer this dimension automatically
flat.reshape(-1, 4): (3, 4)   (12/4 = 3 rows inferred)
flat.reshape(2, -1): (2, 6)   (12/2 = 6 cols inferred)


### 📝 Exercise 2.1 — NumPy Essentials

A dataset has 5 students, each with 3 exam scores (out of 100).

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 2.1
# ─────────────────────────────────────────────────────────────────────────────

scores = np.array([
    [72, 85, 90],   # student 1
    [58, 63, 70],   # student 2
    [91, 88, 95],   # student 3
    [45, 52, 60],   # student 4
    [80, 76, 83],   # student 5
])

# Task 1: Print the shape of the array
# Your code here:

# Task 2: Compute each student's average score (mean across columns, axis=1)
# Your code here:

# Task 3: Find all students whose average score is >= 75 (use boolean indexing)
# Your code here:

# Task 4: Normalise all scores to the range [0, 1] by dividing by 100
# Your code here:

# See the Answer Key at the end of this chapter.

---
# 2.2 pandas for Data Work

**pandas** is the standard library for loading, exploring, and cleaning tabular data. While NumPy works with anonymous rows and columns of numbers, pandas adds **named columns**, **row labels (index)**, and **mixed data types** — making it ideal for real-world datasets.

**Table 2.2 — Core pandas objects**

| Object | What it is | Analogy |
|--------|-----------|--------|
| `Series` | 1-D labelled array — one column of data | A single column in a spreadsheet |
| `DataFrame` | 2-D labelled table — rows and named columns | A full spreadsheet / SQL table |

We will use the **UCI Adult Income** dataset throughout this section. It contains census data for 48,842 individuals; the task is to predict whether annual income exceeds $50K.

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.2a — Loading the UCI Adult Income dataset
# Direct URL — no download or Kaggle account required.
# ─────────────────────────────────────────────────────────────────────────────

# Column names (the original file has no header row)
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

df = pd.read_csv(
    url,
    header=None,            # no header row in this file
    names=column_names,     # assign our own column names
    na_values=' ?',         # treat ' ?' as missing value
    skipinitialspace=True   # strip leading spaces from values
)

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape}  ← ({df.shape[0]:,} rows, {df.shape[1]} columns)")
print()
print("First 5 rows:")
df.head()

Dataset loaded successfully.
Shape: (32561, 15)  ← (32,561 rows, 15 columns)

First 5 rows:


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.2b — First look: data types and missing values
# The first thing you do with any new dataset.
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 55)
print("Column data types")
print("=" * 55)
print(df.dtypes)
print()

print("=" * 55)
print("Missing values per column")
print("=" * 55)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
print(missing_summary[missing_summary['missing_count'] > 0])
print()

print("=" * 55)
print("Numeric column statistics")
print("=" * 55)
df.describe().round(2)

Column data types
age                int64
workclass         object
fnlwgt             int64
education         object
education_num      int64
marital_status    object
occupation        object
relationship      object
race              object
sex               object
capital_gain       int64
capital_loss       int64
hours_per_week     int64
native_country    object
income            object
dtype: object

Missing values per column
Empty DataFrame
Columns: [missing_count, missing_%]
Index: []

Numeric column statistics


,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,32561.00,32561.00,32561.00,32561.00,32561.00,32561.00
mean,38.58,189778.37,10.08,1077.65,87.30,40.44
std,13.64,105549.98,2.57,7385.29,402.96,12.35
min,17.00,12285.00,1.00,0.00,0.00,1.00
25%,28.00,117827.00,9.00,0.00,0.00,40.00
50%,37.00,178356.00,10.00,0.00,0.00,40.00
75%,48.00,237051.00,12.00,0.00,0.00,45.00
max,90.00,1484705.00,16.00,99999.00,4356.00,99.00


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.2c — Selecting columns and filtering rows
# ─────────────────────────────────────────────────────────────────────────────

# Selecting a single column → returns a Series
ages = df['age']
print("Single column (Series):")
print(f"  Type  : {type(ages).__name__}")
print(f"  Shape : {ages.shape}")
print(f"  First 5 values: {ages.head().tolist()}")
print()

# Selecting multiple columns → returns a DataFrame
numeric_cols = df[['age', 'education_num', 'hours_per_week', 'income']]
print("Multiple columns (DataFrame):")
print(numeric_cols.head())
print()

# Filtering rows with boolean conditions
high_earners = df[df['income'] == '>50K']
print(f"High earners (>50K): {len(high_earners):,} rows out of {len(df):,} total")
print(f"That is {len(high_earners)/len(df)*100:.1f}% of the dataset")
print()

# Combining multiple conditions with & (and) | (or)
# Important: each condition must be wrapped in parentheses
senior_high_earners = df[(df['age'] >= 50) & (df['income'] == '>50K')]
print(f"Age >= 50 AND income >50K: {len(senior_high_earners):,} rows")

Single column (Series):
  Type  : Series
  Shape : (32561,)
  First 5 values: [39, 50, 38, 53, 28]

Multiple columns (DataFrame):
   age  education_num  hours_per_week income
0   39             13              40  <=50K
1   50             13              13  <=50K
2   38              9              40  <=50K
3   53              7              40  <=50K
4   28             13              40  <=50K

High earners (>50K): 7,841 rows out of 32,561 total
That is 24.1% of the dataset

Age >= 50 AND income >50K: 2,359 rows


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.2d — Value counts, groupby, and aggregation
# Understanding distributions and group-level summaries.
# ─────────────────────────────────────────────────────────────────────────────

print("Income class distribution:")
print(df['income'].value_counts())
print()

print("Top 5 occupations:")
print(df['occupation'].value_counts().head())
print()

# groupby: split data into groups, apply a function, combine results
# This is the most powerful summarisation tool in pandas
print("Average age and hours_per_week by income class:")
group_summary = df.groupby('income')[['age', 'hours_per_week', 'education_num']].mean().round(1)
print(group_summary)
print()

print("Income rate by education level (top 6):")
edu_income = df.groupby('education')['income'].apply(
    lambda x: (x == '>50K').mean() * 100
).sort_values(ascending=False).round(1).head(6)
print(edu_income.to_string())

Income class distribution:
income
<=50K    24720
>50K      7841
Name: count, dtype: int64

Top 5 occupations:
occupation
Prof-specialty     4140
Craft-repair       4099
Exec-managerial    4066
Adm-clerical       3770
Sales              3650
Name: count, dtype: int64

Average age and hours_per_week by income class:
         age  hours_per_week  education_num
income                                     
<=50K   36.8            38.8            9.6
>50K    44.2            45.5           11.6

Income rate by education level (top 6):
education
Doctorate      74.1
Prof-school    73.4
Masters        55.7
Bachelors      41.5
Assoc-voc      26.1
Assoc-acdm     24.8


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.2e — Handling missing values and creating new columns
# ─────────────────────────────────────────────────────────────────────────────

print(f"Rows before dropping missing values : {len(df):,}")

# Drop rows with any missing value
df_clean = df.dropna()
print(f"Rows after  dropping missing values : {len(df_clean):,}")
print(f"Rows dropped                        : {len(df) - len(df_clean):,}")
print()

# Create a new column from existing ones
# Binary income label: 1 = earns >50K, 0 = earns <=50K
df_clean = df_clean.copy()   # avoid SettingWithCopyWarning
df_clean['income_label'] = (df_clean['income'] == '>50K').astype(int)

print("New 'income_label' column (first 10 values):")
print(df_clean[['income', 'income_label']].head(10).to_string(index=False))
print()
print(f"Label distribution:")
print(df_clean['income_label'].value_counts().to_string())
print("  (0 = <=50K,  1 = >50K)")

Rows before dropping missing values : 32,561
Rows after  dropping missing values : 32,561
Rows dropped                        : 0

New 'income_label' column (first 10 values):
income  income_label
 <=50K             0
 <=50K             0
 <=50K             0
 <=50K             0
 <=50K             0
 <=50K             0
 <=50K             0
  >50K             1
  >50K             1
  >50K             1

Label distribution:
income_label
0    24720
1     7841
  (0 = <=50K,  1 = >50K)


### 📝 Exercise 2.2 — pandas

Using the `df_clean` DataFrame loaded above:

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 2.2
# ─────────────────────────────────────────────────────────────────────────────

# Task 1: How many people work more than 50 hours per week?
# Your code here:

# Task 2: What is the average capital_gain for each income class?
# (use groupby)
# Your code here:

# Task 3: What percentage of self-employed workers (workclass contains 'Self-emp')
# earn more than $50K? Hint: use str.contains() on the workclass column.
# Your code here:

# See the Answer Key at the end of this chapter.

---
# 2.3 PyTorch Tensors

A **tensor** is the fundamental data structure in PyTorch — it is to deep learning what the ndarray is to NumPy. In fact, tensors and NumPy arrays are so similar that they can share the same memory.

The key differences between a PyTorch tensor and a NumPy array are:

**Table 2.3 — NumPy array vs PyTorch tensor**

| | NumPy ndarray | PyTorch tensor |
|--|--------------|----------------|
| Primary use | General numerical computing | Neural network operations |
| GPU support | No | Yes — `.to('cuda')` or `.to('mps')` |
| Gradient tracking | No | Yes — `requires_grad=True` enables autograd |
| Interoperability | `.values` from pandas, direct conversion to tensor | `.numpy()` to convert back to NumPy |
| Default dtype | `float64` | `float32` (preferred for neural networks) |

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.3a — Creating tensors
# ─────────────────────────────────────────────────────────────────────────────

# From a Python list
t1 = torch.tensor([1.0, 2.0, 3.0, 4.0])
print("From list:")
print(f"  Values : {t1}")
print(f"  Shape  : {t1.shape}")
print(f"  dtype  : {t1.dtype}")
print()

# From a 2-D list (like a batch of samples)
t2 = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])
print("2-D tensor:")
print(t2)
print(f"  Shape: {t2.shape}   ← torch.Size([2, 3]) = 2 rows, 3 columns")
print()

# Special constructors
print("torch.zeros((2, 3))  :"); print(torch.zeros((2, 3)))
print("torch.ones((2, 3))   :"); print(torch.ones((2, 3)))
print("torch.randn((2, 3))  :"); print(torch.randn((2, 3)).round(decimals=4))
print()

# Specifying dtype explicitly
# float32 is the standard for neural network weights
t_float = torch.tensor([1, 2, 3], dtype=torch.float32)
t_int   = torch.tensor([1, 2, 3], dtype=torch.int64)
print(f"float32 tensor: {t_float}  dtype={t_float.dtype}")
print(f"int64   tensor: {t_int}    dtype={t_int.dtype}")

From list:
  Values : tensor([1., 2., 3., 4.])
  Shape  : torch.Size([4])
  dtype  : torch.float32

2-D tensor:
tensor([[1., 2., 3.],
        [4., 5., 6.]])
  Shape: torch.Size([2, 3])   ← torch.Size([2, 3]) = 2 rows, 3 columns

torch.zeros((2, 3))  :
tensor([[0., 0., 0.],
        [0., 0., 0.]])
torch.ones((2, 3))   :
tensor([[1., 1., 1.],
        [1., 1., 1.]])
torch.randn((2, 3))  :
tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863]])

float32 tensor: tensor([1., 2., 3.])  dtype=torch.float32
int64   tensor: tensor([1, 2, 3])    dtype=torch.int64


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.3b — Tensor operations
# Tensor arithmetic works identically to NumPy — element-wise by default.
# ─────────────────────────────────────────────────────────────────────────────

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("Element-wise operations:")
print(f"  a + b  = {a + b}")
print(f"  a * b  = {a * b}")
print(f"  a ** 2 = {a ** 2}")
print()

# Aggregate operations
print("Aggregations on a tensor:")
print(f"  a.sum()  = {a.sum().item():.1f}")
print(f"  a.mean() = {a.mean().item():.4f}")
print(f"  a.max()  = {a.max().item():.1f}")
print()

# .item() extracts a single Python number from a scalar tensor
# You will see this constantly when printing loss values during training
loss_tensor = torch.tensor(0.4231)
print(f"  loss_tensor        = {loss_tensor}   (type: {type(loss_tensor).__name__})")
print(f"  loss_tensor.item() = {loss_tensor.item():.4f}  (type: {type(loss_tensor.item()).__name__})")

Element-wise operations:
  a + b  = tensor([5., 7., 9.])
  a * b  = tensor([ 4., 10., 18.])
  a ** 2 = tensor([1., 4., 9.])

Aggregations on a tensor:
  a.sum()  = 6.0
  a.mean() = 2.0000
  a.max()  = 3.0

  loss_tensor        = 0.42309999465942383   (type: Tensor)
  loss_tensor.item() = 0.4231  (type: float)


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.3c — Shape manipulation: reshape, view, squeeze, unsqueeze
#
# Reshaping tensors is one of the most common sources of bugs in DL code.
# Understanding shape transformations will save you hours of debugging.
# ─────────────────────────────────────────────────────────────────────────────

t = torch.arange(12, dtype=torch.float32)   # [0, 1, 2, ..., 11]
print(f"Original shape: {t.shape}")
print()

# view: reshape without copying data (most efficient)
# -1 means: infer this dimension
t_2d = t.view(3, 4)
print(f"view(3, 4)   shape: {t_2d.shape}")
print(t_2d)
print()

t_auto = t.view(-1, 4)    # PyTorch works out that 12/4 = 3 rows
print(f"view(-1, 4)  shape: {t_auto.shape}   (-1 inferred as 3)")
print()

# unsqueeze: add a dimension of size 1 at the specified position
# Common when a model expects a batch dimension but you have one sample
single_sample = torch.tensor([0.5, 0.3, 0.9])
print(f"single_sample shape           : {single_sample.shape}     (3,)")
batched = single_sample.unsqueeze(0)            # add batch dim at position 0
print(f"after unsqueeze(0) shape      : {batched.shape}   (1, 3) — now looks like batch of 1")
print()

# squeeze: remove dimensions of size 1
print(f"after squeeze()    shape      : {batched.squeeze().shape}     back to (3,)")

Original shape: torch.Size([12])

view(3, 4)   shape: torch.Size([3, 4])
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])

view(-1, 4)  shape: torch.Size([3, 4])   (-1 inferred as 3)

single_sample shape           : torch.Size([3])     (3,)
after unsqueeze(0) shape      : torch.Size([1, 3])   (1, 3) — now looks like batch of 1

after squeeze()    shape      : torch.Size([3])     back to (3,)


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.3d — Device placement: CPU, CUDA, MPS
#
# PyTorch can run computations on different hardware.
# All tensors involved in an operation must be on the same device.
# ─────────────────────────────────────────────────────────────────────────────

# Detect the best available device
if torch.cuda.is_available():
    device = torch.device('cuda')       # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device('mps')        # Apple Silicon GPU
else:
    device = torch.device('cpu')        # CPU fallback

print(f"Device selected: {device}")
print()

# Create a tensor and move it to the selected device
t = torch.randn(3, 4)
t_on_device = t.to(device)

print(f"Tensor device before .to(): {t.device}")
print(f"Tensor device after  .to(): {t_on_device.device}")
print()
print("In every subsequent chapter, you will see this pattern at the top of training loops:")
print("  model = model.to(device)")
print("  X     = X.to(device)")
print("  y     = y.to(device)")
print("This ensures all computation happens on the same hardware.")

Device selected: cpu

Tensor device before .to(): cpu
Tensor device after  .to(): cpu

In every subsequent chapter, you will see this pattern at the top of training loops:
  model = model.to(device)
  X     = X.to(device)
  y     = y.to(device)
This ensures all computation happens on the same hardware.


### 📝 Exercise 2.3 — PyTorch Tensors

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 2.3
# ─────────────────────────────────────────────────────────────────────────────

# Task 1: Create a tensor of shape (4, 3) filled with random values from
#         a normal distribution, using dtype float32
# Your code here:

# Task 2: Reshape it to (2, 6) using .view()
# Your code here:

# Task 3: Compute and print the mean and standard deviation of all elements
# Your code here:

# Task 4: Add an unsqueeze(0) to simulate a batch dimension.
#         Print the shape before and after.
# Your code here:

# See the Answer Key at the end of this chapter.

> **A note on 3-D and 4-D tensors**
>
> Everything covered in this section — shapes, indexing, `.view()`, `.unsqueeze()`, device placement — applies equally to tensors with more dimensions. You have only seen 1-D and 2-D tensors so far because that is all tabular data needs. Higher-dimensional tensors appear later when the data demands it:
>
> | Shape | Dimensions | Chapter | What it represents |
> |-------|-----------|---------|-------------------|
> | `(batch, features)` | 2-D | 2, 4 | A batch of tabular samples |
> | `(batch, seq_len, features)` | 3-D | 6 — RNNs / LSTMs | A batch of sequences (time series, text) |
> | `(batch, channels, height, width)` | 4-D | 5 — CNNs | A batch of images |
>
> When you reach Chapters 5 and 6, the tensor shapes will look unfamiliar at first. Come back to this table — the same rules apply.

---
# 2.4 Connecting the Three

In practice, data flows through all three libraries in a typical deep learning project:

1. **pandas** — load raw data from CSV, clean it, handle missing values, encode categories
2. **NumPy** — numerical operations, array manipulations, interfacing with scikit-learn
3. **PyTorch tensors** — feed data into neural networks, move to GPU, compute gradients

**Table 2.4 — Conversion reference**

| From | To | Method | Notes |
|------|----|--------|-------|
| pandas Series / DataFrame | NumPy array | `.values` or `.to_numpy()` | Returns a view where possible |
| NumPy array | PyTorch tensor | `torch.from_numpy(arr)` | **Shares memory** — changes to one affect the other |
| NumPy array | PyTorch tensor | `torch.tensor(arr)` | **Copies data** — independent |
| PyTorch tensor | NumPy array | `.numpy()` | Tensor must be on CPU and not require grad |
| PyTorch tensor | Python scalar | `.item()` | Only for single-element tensors |
| pandas DataFrame | PyTorch tensor | `torch.tensor(df.values, dtype=torch.float32)` | Most common pattern in data loading |

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.4a — pandas → NumPy → PyTorch
# The standard path data takes before reaching a neural network.
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: Select numeric columns from our DataFrame
numeric_features = ['age', 'education_num', 'hours_per_week',
                    'capital_gain', 'capital_loss']

df_numeric = df_clean[numeric_features]
print(f"Step 1 — pandas DataFrame:")
print(f"  Type : {type(df_numeric).__name__}")
print(f"  Shape: {df_numeric.shape}")
print()

# Step 2: Convert to NumPy
arr = df_numeric.to_numpy()         # same as df_numeric.values
print(f"Step 2 — NumPy array:")
print(f"  Type : {type(arr).__name__}")
print(f"  Shape: {arr.shape}")
print(f"  dtype: {arr.dtype}")
print()

# Step 3: Convert to PyTorch tensor
# Always cast to float32 — this is what neural network layers expect
tensor = torch.tensor(arr, dtype=torch.float32)
print(f"Step 3 — PyTorch tensor:")
print(f"  Type : {type(tensor).__name__}")
print(f"  Shape: {tensor.shape}")
print(f"  dtype: {tensor.dtype}")
print(f"  First row: {tensor[0]}")

Step 1 — pandas DataFrame:
  Type : DataFrame
  Shape: (32561, 5)

Step 2 — NumPy array:
  Type : ndarray
  Shape: (32561, 5)
  dtype: int64

Step 3 — PyTorch tensor:
  Type : Tensor
  Shape: torch.Size([32561, 5])
  dtype: torch.float32
  First row: tensor([  39.,   13.,   40., 2174.,    0.])


In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.4b — Memory sharing: torch.from_numpy vs torch.tensor
#
# This is an important subtlety that can cause silent bugs if misunderstood.
# ─────────────────────────────────────────────────────────────────────────────

original_array = np.array([1.0, 2.0, 3.0])

# from_numpy: shares memory — modifying one modifies the other
shared_tensor  = torch.from_numpy(original_array)

# torch.tensor: copies data — completely independent
copied_tensor  = torch.tensor(original_array)

print("Before modification:")
print(f"  original_array : {original_array}")
print(f"  shared_tensor  : {shared_tensor}")
print(f"  copied_tensor  : {copied_tensor}")
print()

# Modify the original NumPy array
original_array[0] = 99.0

print("After setting original_array[0] = 99:")
print(f"  original_array : {original_array}")
print(f"  shared_tensor  : {shared_tensor}   ← CHANGED (shared memory)")
print(f"  copied_tensor  : {copied_tensor}   ← unchanged (independent copy)")
print()
print("Rule of thumb:")
print("  Use torch.from_numpy()  when you want efficiency and won't modify the array.")
print("  Use torch.tensor()      when you want a fully independent copy.")

Before modification:
  original_array : [1. 2. 3.]
  shared_tensor  : tensor([1., 2., 3.], dtype=torch.float64)
  copied_tensor  : tensor([1., 2., 3.], dtype=torch.float64)

After setting original_array[0] = 99:
  original_array : [99.  2.  3.]
  shared_tensor  : tensor([99.,  2.,  3.], dtype=torch.float64)   ← CHANGED (shared memory)
  copied_tensor  : tensor([1., 2., 3.], dtype=torch.float64)   ← unchanged (independent copy)

Rule of thumb:
  Use torch.from_numpy()  when you want efficiency and won't modify the array.
  Use torch.tensor()      when you want a fully independent copy.


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.4c — Converting back: tensor → NumPy
# ─────────────────────────────────────────────────────────────────────────────

t = torch.tensor([10.0, 20.0, 30.0])

# .numpy() — converts to NumPy array
# Requirements: tensor must be on CPU, must not require gradient
arr_back = t.numpy()
print(f"Tensor → NumPy: {arr_back}  (type: {type(arr_back).__name__})")
print()

# If the tensor requires gradients (as model outputs do), detach first
t_with_grad = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
arr_detached = t_with_grad.detach().numpy()
print(f"Tensor with grad → NumPy (via .detach()): {arr_detached}")
print()
print("You will use .detach().numpy() often when extracting predictions")
print("from a trained model to compute metrics or plot results.")

Tensor → NumPy: [10. 20. 30.]  (type: ndarray)

Tensor with grad → NumPy (via .detach()): [1. 2. 3.]

You will use .detach().numpy() often when extracting predictions
from a trained model to compute metrics or plot results.


---
# 2.5 Linear Algebra Essentials

You do not need to be a linear algebra expert to work through this book. But three operations appear in every neural network, so understanding them will make the code much clearer.

**Table 2.5 — The three operations you must know**

| Operation | Notation | PyTorch | What it does |
|-----------|----------|---------|-------------|
| Dot product | $\mathbf{a} \cdot \mathbf{b}$ | `torch.dot(a, b)` | Multiply two vectors element-wise, then sum — produces a scalar |
| Matrix multiplication | $AB$ | `A @ B` or `torch.matmul(A, B)` | Generalises dot product to 2-D arrays — key operation in every layer |
| Element-wise multiply | $A \odot B$ | `A * B` | Multiply corresponding elements — requires same shape |

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.5a — Dot product
#
# The dot product is literally what a single neuron computes:
#   weighted_sum = inputs · weights
# ─────────────────────────────────────────────────────────────────────────────

inputs  = torch.tensor([0.5, 0.8, 0.3])   # one customer's 3 features
weights = torch.tensor([0.4, 0.9, -0.2])  # learned weights for one neuron
bias    = torch.tensor(0.1)

# Dot product: multiply each input by its weight, sum everything
weighted_sum = torch.dot(inputs, weights) + bias

print("Dot product — one neuron's computation:")
for i, (x, w) in enumerate(zip(inputs.tolist(), weights.tolist())):
    print(f"  input[{i}] × weight[{i}] = {x} × {w} = {x*w:.4f}")
print(f"  + bias = {bias.item()}")
print(f"  ─────────────────────")
print(f"  weighted_sum = {weighted_sum.item():.4f}")
print()
print("This is exactly torch.dot(inputs, weights) + bias")

Dot product — one neuron's computation:
  input[0] × weight[0] = 0.5 × 0.4000000059604645 = 0.2000
  input[1] × weight[1] = 0.800000011920929 × 0.8999999761581421 = 0.7200
  input[2] × weight[2] = 0.30000001192092896 × -0.20000000298023224 = -0.0600
  + bias = 0.10000000149011612
  ─────────────────────
  weighted_sum = 0.9600

This is exactly torch.dot(inputs, weights) + bias


In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.5b — Matrix multiplication
#
# A fully-connected layer in PyTorch is: output = input @ weight.T + bias
# Understanding @ (matmul) is essential for understanding layer dimensions.
#
# Shape rule: (m, k) @ (k, n) → (m, n)
#   The inner dimensions must match; the outer dimensions give the output shape.
# ─────────────────────────────────────────────────────────────────────────────

# Batch of 4 samples, each with 3 features
X = torch.randn(4, 3)   # shape: (4, 3)

# Weight matrix for a layer with 3 inputs → 5 outputs
W = torch.randn(3, 5)   # shape: (3, 5)

# Output: 4 samples, each now has 5 values (one per output neuron)
output = X @ W          # shape rule: (4, 3) @ (3, 5) → (4, 5)

print("Matrix multiplication — a fully-connected layer:")
print(f"  X shape      : {X.shape}   (4 samples, 3 features)")
print(f"  W shape      : {W.shape}   (3 inputs, 5 output neurons)")
print(f"  X @ W shape  : {output.shape}  (4 samples, 5 output values)")
print()
print("Shape rule for matmul: (m, k) @ (k, n) → (m, n)")
print("The INNER dimensions (k) must match. The OUTER dimensions (m, n) give the result shape.")
print()

# Demonstrating a shape mismatch error
print("What happens with a shape mismatch:")
try:
    bad_W = torch.randn(4, 5)   # wrong: inner dim is 4, but X has 3 columns
    _ = X @ bad_W
except RuntimeError as e:
    print(f"  RuntimeError: {e}")
    print("  Fix: ensure X.shape[-1] == W.shape[0]")

Matrix multiplication — a fully-connected layer:
  X shape      : torch.Size([4, 3])   (4 samples, 3 features)
  W shape      : torch.Size([3, 5])   (3 inputs, 5 output neurons)
  X @ W shape  : torch.Size([4, 5])  (4 samples, 5 output values)

Shape rule for matmul: (m, k) @ (k, n) → (m, n)
The INNER dimensions (k) must match. The OUTER dimensions (m, n) give the result shape.

What happens with a shape mismatch:
  RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x3 and 4x5)
  Fix: ensure X.shape[-1] == W.shape[0]


In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.5c — Broadcasting rules
#
# Broadcasting lets you operate on tensors of different shapes
# without manually expanding them. PyTorch follows NumPy broadcasting rules.
#
# Rule: dimensions are compared from the right.
#       A dimension of size 1 can be stretched to match the other.
# ─────────────────────────────────────────────────────────────────────────────

print("Broadcasting examples:")
print()

# (3,) + scalar
a = torch.tensor([1.0, 2.0, 3.0])
print(f"  (3,) + scalar 10  → {a + 10}")

# (4, 3) + (3,)  — bias addition in a layer
output = torch.randn(4, 3)
bias   = torch.tensor([0.1, 0.2, 0.3])   # one bias per neuron
result = output + bias
print(f"  (4,3) + (3,) → {result.shape}  (bias added to every row)")
print()

print("Broadcasting shape alignment (right-aligned):")
print("  Tensor A:   (4, 3)")
print("  Tensor B:      (3)")
print("  Comparison: 4≠? | 3=3  ← rightmost dims match")
print("  B is stretched from (3) to (4, 3) — result shape: (4, 3)")
print()

# Shape that CANNOT broadcast
print("Shape that cannot broadcast:")
try:
    bad = torch.randn(4, 3) + torch.randn(2, 3)
except RuntimeError as e:
    print(f"  RuntimeError — 4 and 2 are both > 1 and don't match.")

Broadcasting examples:

  (3,) + scalar 10  → tensor([11., 12., 13.])
  (4,3) + (3,) → torch.Size([4, 3])  (bias added to every row)

Broadcasting shape alignment (right-aligned):
  Tensor A:   (4, 3)
  Tensor B:      (3)
  Comparison: 4≠? | 3=3  ← rightmost dims match
  B is stretched from (3) to (4, 3) — result shape: (4, 3)

Shape that cannot broadcast:
  RuntimeError — 4 and 2 are both > 1 and don't match.


---
# 2.6 Data Pipelines

Neural networks do not train on the full dataset at once. They process small **batches** — typically 32 to 256 samples — and update weights after each batch. This requires an efficient mechanism to:

1. Wrap the data so it can be indexed sample by sample
2. Shuffle the data at the start of each epoch (so the model doesn't memorise order)
3. Yield batches of the right size with minimal CPU overhead

PyTorch handles all of this with two classes: **`Dataset`** and **`DataLoader`**.

**Table 2.6 — Dataset vs DataLoader**

| Class | Responsibility | Key methods / args |
|-------|---------------|-------------------|
| `Dataset` | Stores data; defines how to retrieve one sample by index | `__len__()`, `__getitem__(idx)` |
| `DataLoader` | Wraps a Dataset; yields shuffled batches during training | `batch_size`, `shuffle`, `num_workers` |

In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.6a — Building a custom Dataset for the Adult Income data
#
# Every Dataset subclass must implement exactly two methods:
#   __len__   : returns the total number of samples
#   __getitem__: returns one (features, label) pair given an index
# ─────────────────────────────────────────────────────────────────────────────

from torch.utils.data import Dataset, DataLoader

class AdultIncomeDataset(Dataset):
    """PyTorch Dataset wrapping the Adult Income feature matrix and labels.

    Args:
        features_array: NumPy array of shape (n_samples, n_features)
        labels_array:   NumPy array of shape (n_samples,)
    """

    def __init__(self, features_array, labels_array):
        # Convert to float32 tensors once at construction time
        # (not inside __getitem__ — that would be slow)
        self.X = torch.tensor(features_array, dtype=torch.float32)
        self.y = torch.tensor(labels_array,   dtype=torch.float32)

    def __len__(self):
        # DataLoader calls this to know how many samples exist
        return len(self.X)

    def __getitem__(self, idx):
        # DataLoader calls this for each sample index in a batch
        return self.X[idx], self.y[idx]


# --- Build a minimal feature matrix for demonstration ---
# (Full preprocessing happens in Section 2.7)
demo_features = df_clean[['age', 'education_num', 'hours_per_week']].to_numpy().astype('float32')
demo_labels   = df_clean['income_label'].to_numpy().astype('float32')

dataset = AdultIncomeDataset(demo_features, demo_labels)

print(f"Dataset length (total samples) : {len(dataset):,}")
print()

# Access a single sample by index
features_0, label_0 = dataset[0]
print(f"Sample 0:")
print(f"  Features : {features_0}   (age, education_num, hours_per_week)")
print(f"  Label    : {label_0.item()}   (0 = <=50K, 1 = >50K)")

Dataset length (total samples) : 32,561

Sample 0:
  Features : tensor([39., 13., 40.])   (age, education_num, hours_per_week)
  Label    : 0.0   (0 = <=50K, 1 = >50K)


In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.6b — Wrapping with DataLoader and iterating batches
# ─────────────────────────────────────────────────────────────────────────────

# DataLoader wraps the Dataset and handles batching + shuffling automatically
train_loader = DataLoader(
    dataset,
    batch_size=32,    # process 32 samples at a time
    shuffle=True,     # shuffle at the start of every epoch
    drop_last=False   # keep the final smaller batch if dataset size is not divisible
)

total_batches = len(train_loader)   # == ceil(len(dataset) / batch_size)
print(f"Dataset size  : {len(dataset):,} samples")
print(f"Batch size    : 32")
print(f"Total batches : {total_batches}  (ceil({len(dataset):,} / 32))")
print()

# Iterate one epoch — this is the loop structure used in every training chapter
print("Shape of the first 3 batches:")
for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
    print(f"  Batch {batch_idx}: X shape = {X_batch.shape},  y shape = {y_batch.shape}")
    if batch_idx == 2:
        print("  ... (remaining batches omitted)")
        break

print()
print("In a training loop you will write:")
print("  for epoch in range(num_epochs):")
print("      for X_batch, y_batch in train_loader:")
print("          # forward pass, compute loss, backprop, update weights")

Dataset size  : 32,561 samples
Batch size    : 32
Total batches : 1018  (ceil(32,561 / 32))

Shape of the first 3 batches:
  Batch 0: X shape = torch.Size([32, 3]),  y shape = torch.Size([32])
  Batch 1: X shape = torch.Size([32, 3]),  y shape = torch.Size([32])
  Batch 2: X shape = torch.Size([32, 3]),  y shape = torch.Size([32])
  ... (remaining batches omitted)

In a training loop you will write:
  for epoch in range(num_epochs):
      for X_batch, y_batch in train_loader:
          # forward pass, compute loss, backprop, update weights


---
# 2.7 Normalisation & Preprocessing

Raw data is almost never in the right form for a neural network. Two problems are especially common:

1. **Scale differences** — one feature ranges from 0 to 1 (binary flag), another from 0 to 500,000 (income). The large-scale feature will dominate gradient updates, making training unstable or very slow.

2. **Categorical variables** — neural networks require numbers. Text categories like `"Male"`, `"Female"` or `"Private"`, `"Self-emp"` must be converted to numeric form before being fed to a model.

**Table 2.7a — Normalisation methods**

| Method | Formula | Result range | When to use |
|--------|---------|-------------|-------------|
| **StandardScaler** (Z-score) | $(x - \mu) / \sigma$ | Mean 0, std 1 (unbounded) | Most cases — especially with normally distributed features |
| **MinMaxScaler** | $(x - x_{min}) / (x_{max} - x_{min})$ | [0, 1] | When you need a bounded range, e.g. for image pixels |

> **Important:** Apply scaling only to **continuous numeric columns**. One-hot encoded and label-encoded columns are already binary (0 or 1) — scaling them destroys their meaning and is incorrect.

**Table 2.7b — Categorical encoding methods**

| Method | Output | When to use |
|--------|--------|-------------|
| **Label encoding** | Single integer (0, 1, 2, ...) | Binary categories (yes/no, male/female), or ordinal categories |
| **One-hot encoding** | N binary columns, one per category | Nominal categories with no natural order (occupation, country) |

In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.7a — Why scale matters: a visual demonstration
# ─────────────────────────────────────────────────────────────────────────────

# Simulate two features before and after scaling
sample_data = pd.DataFrame({
    'age':            [25, 35, 45, 55, 65],
    'capital_gain':   [0, 5000, 15000, 50000, 99999],
})

print("Before normalisation:")
print(sample_data.to_string(index=False))
print(f"  age range        : {sample_data['age'].min()} – {sample_data['age'].max()}")
print(f"  capital_gain range: {sample_data['capital_gain'].min()} – {sample_data['capital_gain'].max()}")
print()
print("Problem: capital_gain is ~1000× larger than age.")
print("Gradients for capital_gain weights will dominate training.")
print()

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled = scaler.fit_transform(sample_data)
scaled_df = pd.DataFrame(scaled, columns=sample_data.columns).round(3)

print("After StandardScaler (Z-score normalisation):")
print(scaled_df.to_string(index=False))
print(f"  age mean: {scaled_df['age'].mean():.2f}, std: {scaled_df['age'].std():.2f}")
print(f"  capital_gain mean: {scaled_df['capital_gain'].mean():.2f}, std: {scaled_df['capital_gain'].std():.2f}")
print()
print("Both features now have comparable scale — training will be stable.")

Before normalisation:
 age  capital_gain
  25             0
  35          5000
  45         15000
  55         50000
  65         99999
  age range        : 25 – 65
  capital_gain range: 0 – 99999

Problem: capital_gain is ~1000× larger than age.
Gradients for capital_gain weights will dominate training.

After StandardScaler (Z-score normalisation):
   age  capital_gain
-1.414        -0.911
-0.707        -0.777
 0.000        -0.509
 0.707         0.429
 1.414         1.768
  age mean: -0.00, std: 1.12
  capital_gain mean: 0.00, std: 1.12

Both features now have comparable scale — training will be stable.


In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.7b — Full preprocessing pipeline for Adult Income
#
# Correct approach: treat numeric and categorical columns SEPARATELY.
#
#   Numeric columns     → StandardScaler (continuous values need normalisation)
#   Binary categorical  → LabelEncoder   (already 0/1 — no scaling needed)
#   Multi-class categ.  → One-hot encode (already 0/1 flags — no scaling needed)
#
# Why NOT scale encoded columns:
#   One-hot and label-encoded columns are binary flags (0 or 1).
#   Scaling them would produce negative values and destroy their meaning.
#   The network handles 0/1 indicator columns well at their natural scale.
#
# Steps:
#   1. Separate columns: numeric | binary categorical | multi-class categorical
#   2. Scale numeric columns with StandardScaler
#   3. Label-encode binary categorical columns
#   4. One-hot encode multi-class categorical columns
#   5. Concatenate all three parts into one feature matrix
#   6. Split into train / validation / test sets
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

data = df_clean.copy()

# ── Column definitions ────────────────────────────────────────────────────────
# Continuous numeric features — these have meaningful magnitude and spread
numeric_cols = ['age', 'education_num', 'hours_per_week', 'capital_gain', 'capital_loss']

# Binary categorical features — only 2 unique values each
binary_cat_cols = ['sex']

# Multi-class categorical features — 3 or more unique values
multi_cat_cols = ['workclass', 'education', 'marital_status',
                  'occupation', 'relationship', 'race', 'native_country']

# Label column (target) — kept separate from features
label_col = 'income'

# ── Step 1: Extract the label vector ─────────────────────────────────────────
le_label = LabelEncoder()
y = le_label.fit_transform(data[label_col]).astype('float32')   # <=50K → 0, >50K → 1
print(f"Label encoding: {le_label.classes_} → {le_label.transform(le_label.classes_)}")
print()

# ── Step 2: Process numeric columns — extract as NumPy, scale later ──────────
X_numeric = data[numeric_cols].to_numpy().astype('float32')
print(f"Numeric block shape         : {X_numeric.shape}  ← will be scaled")

# ── Step 3: Label-encode binary categorical columns ───────────────────────────
# Result is a 2-D array of 0/1 values — one column per binary feature
binary_encoded = np.column_stack([
    LabelEncoder().fit_transform(data[col])
    for col in binary_cat_cols
]).astype('float32')
print(f"Binary categorical block shape: {binary_encoded.shape}  ← already 0/1, no scaling")

# ── Step 4: One-hot encode multi-class categorical columns ────────────────────
# drop_first=True drops one redundant column per feature to avoid multicollinearity
ohe_df = pd.get_dummies(data[multi_cat_cols], drop_first=True)
X_ohe  = ohe_df.to_numpy().astype('float32')
print(f"One-hot block shape           : {X_ohe.shape}  ← already 0/1, no scaling")
print()

# ── Step 5: Train / val / test split BEFORE scaling ──────────────────────────
# Split indices consistently across all three blocks
n = len(y)
idx = np.arange(n)
idx_train, idx_temp = train_test_split(idx, test_size=0.30, random_state=42, stratify=y)
idx_val,   idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42, stratify=y[idx_temp])

print(f"Split sizes — train: {len(idx_train):,}  val: {len(idx_val):,}  test: {len(idx_test):,}")
print()

# ── Step 6: Scale numeric block only — fit on train, transform all splits ─────
# CRITICAL: fit only on training rows to prevent data leakage
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_numeric[idx_train])   # fit + transform
X_num_val   = scaler.transform(X_numeric[idx_val])         # transform only
X_num_test  = scaler.transform(X_numeric[idx_test])        # transform only

print("Numeric block after StandardScaler:")
print(f"  Train mean (age col): {X_num_train[:, 0].mean():.4f}  (should be ~0)")
print(f"  Train std  (age col): {X_num_train[:, 0].std():.4f}  (should be ~1)")
print()

# ── Step 7: Concatenate all three blocks ──────────────────────────────────────
# Order: scaled numeric | binary 0/1 | one-hot 0/1
X_train = np.concatenate([X_num_train,             binary_encoded[idx_train], X_ohe[idx_train]], axis=1)
X_val   = np.concatenate([X_num_val,               binary_encoded[idx_val],   X_ohe[idx_val]],   axis=1)
X_test  = np.concatenate([X_num_test,              binary_encoded[idx_test],  X_ohe[idx_test]],  axis=1)
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

print("Final feature matrix shapes:")
print(f"  X_train : {X_train.shape}")
print(f"  X_val   : {X_val.shape}")
print(f"  X_test  : {X_test.shape}")
print()
print(f"Column breakdown: {len(numeric_cols)} scaled numeric  +"
      f"  {binary_encoded.shape[1]} binary  +"
      f"  {X_ohe.shape[1]} one-hot  =  {X_train.shape[1]} total features")

Label encoding: ['<=50K' '>50K'] → [0 1]

Numeric block shape         : (32561, 5)  ← will be scaled
Binary categorical block shape: (32561, 1)  ← already 0/1, no scaling
One-hot block shape           : (32561, 93)  ← already 0/1, no scaling

Split sizes — train: 22,792  val: 4,884  test: 4,885

Numeric block after StandardScaler:
  Train mean (age col): -0.0000  (should be ~0)
  Train std  (age col): 1.0000  (should be ~1)

Final feature matrix shapes:
  X_train : (22792, 99)
  X_val   : (4884, 99)
  X_test  : (4885, 99)

Column breakdown: 5 scaled numeric  +  1 binary  +  93 one-hot  =  99 total features


In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.7c — Wrap everything in DataLoaders, ready for Chapter 4
# ─────────────────────────────────────────────────────────────────────────────

from torch.utils.data import Dataset, DataLoader

class AdultIncomeDataset(Dataset):
    """Dataset for the Adult Income classification task."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create Dataset objects
train_dataset = AdultIncomeDataset(X_train, y_train)
val_dataset   = AdultIncomeDataset(X_val,   y_val)
test_dataset  = AdultIncomeDataset(X_test,  y_test)

# Create DataLoaders
# shuffle=True only for training — validation and test order doesn't matter
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print("DataLoaders ready:")
print(f"  train_loader : {len(train_loader):,} batches  ({len(train_dataset):,} samples, batch_size=64)")
print(f"  val_loader   : {len(val_loader):,} batches  ({len(val_dataset):,} samples)")
print(f"  test_loader  : {len(test_loader):,} batches  ({len(test_dataset):,} samples)")
print()

# Confirm batch shapes
X_batch, y_batch = next(iter(train_loader))
print(f"Sample batch:")
print(f"  X_batch shape : {X_batch.shape}   (64 samples, {X_batch.shape[1]} features)")
print(f"  y_batch shape : {y_batch.shape}")
print()
print(f"Number of input features: {X_batch.shape[1]}")
print(f"This is the value you will pass to FFN(input_size=...) in Chapter 4.")

DataLoaders ready:
  train_loader : 357 batches  (22,792 samples, batch_size=64)
  val_loader   : 77 batches  (4,884 samples)
  test_loader  : 77 batches  (4,885 samples)

Sample batch:
  X_batch shape : torch.Size([64, 99])   (64 samples, 99 features)
  y_batch shape : torch.Size([64])

Number of input features: 99
This is the value you will pass to FFN(input_size=...) in Chapter 4.


### 📝 Exercise 2.7 — Preprocessing

Answer the questions below as code comments, then write the code for Tasks 2 and 3.

In [30]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 2.7
# ─────────────────────────────────────────────────────────────────────────────

# Task 1: Why do we fit the scaler on X_train only and not on the full dataset?
#   Answer: ?

# Task 2: A new dataset has 4 numeric features and 1 categorical feature
#         with 5 unique values. After one-hot encoding (drop_first=True),
#         how many columns will the feature matrix have?
#   Answer (calculation): ?

# Task 3: Create a DataLoader from train_dataset (already defined above)
#         with batch_size=128, shuffle=True. Print the number of batches.
# Your code here:

# See the Answer Key at the end of this chapter.

---
## Chapter 2 Summary

| Concept | Key takeaway |
|---------|-------------|
| **NumPy arrays** | Fast n-dimensional number arrays; support vectorised ops and broadcasting |
| **Shape** | The most important property of any array/tensor; always check it |
| **pandas DataFrame** | Named columns, mixed types, powerful groupby and filtering |
| **PyTorch tensor** | Like NumPy but with GPU support and gradient tracking |
| **Conversions** | pandas → `.to_numpy()` → `torch.tensor()` is the standard path |
| **Memory sharing** | `torch.from_numpy()` shares memory; `torch.tensor()` copies |
| **Matrix multiply** | `A @ B`: inner dims must match; outer dims give result shape |
| **Dataset** | Wraps data with `__len__` and `__getitem__`; defines one-sample access |
| **DataLoader** | Wraps Dataset; yields shuffled batches during training |
| **Normalisation** | Always fit scaler on training data only; transform val/test separately |
| **Encoding** | Binary cats → LabelEncoder; multi-class cats → one-hot (pd.get_dummies) |

---

## What Comes Next

Chapter 3 uses everything from this chapter to explore **optimisation and training**:
- How gradient descent works in practice (with different optimisers: SGD, Adam)
- Learning rate and scheduling — why the step size matters
- Common training problems: vanishing gradients, exploding gradients, underfitting, overfitting
- Regularisation techniques: dropout, L2, batch normalisation
- Introducing the **ModelPipeline** — the reusable training and inference wrapper used in every chapter from Chapter 3 onwards

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*

---
## Answer Key — Chapter 2 Exercises

---

### Exercise 2.1 — NumPy Essentials

```python
# Task 1
print(scores.shape)          # (5, 3)

# Task 2
avg_scores = scores.mean(axis=1)
print(avg_scores)            # [82.33, 63.67, 91.33, 52.33, 79.67]

# Task 3
passing = scores[avg_scores >= 75]
print(passing)               # students 1, 3, and 5

# Task 4
normalised = scores / 100
print(normalised)
```

---

### Exercise 2.2 — pandas

```python
# Task 1
print(len(df_clean[df_clean['hours_per_week'] > 50]))

# Task 2
print(df_clean.groupby('income')['capital_gain'].mean().round(2))

# Task 3
self_emp = df_clean[df_clean['workclass'].str.contains('Self-emp', na=False)]
pct = (self_emp['income'] == '>50K').mean() * 100
print(f"{pct:.1f}%")
```

---

### Exercise 2.3 — PyTorch Tensors

```python
# Task 1
t = torch.randn(4, 3, dtype=torch.float32)

# Task 2
t2 = t.view(2, 6)
print(t2.shape)   # torch.Size([2, 6])

# Task 3
print(t.mean().item(), t.std().item())

# Task 4
print(t.shape)              # torch.Size([4, 3])
t_batched = t.unsqueeze(0)
print(t_batched.shape)      # torch.Size([1, 4, 3])
```

---

### Exercise 2.7 — Preprocessing

**Task 1:** If we fit the scaler on the full dataset, the mean and standard deviation are computed using values from the validation and test sets. This leaks information about those sets into the training process — the model indirectly benefits from seeing test data before evaluation. Always fit on training data only, then apply the same transformation to val and test.

Also note: the scaler should be applied **only to numeric columns**. One-hot encoded and label-encoded columns are binary flags (0 or 1) — scaling them would produce negative values and destroy their meaning. The network handles 0/1 indicator columns correctly at their natural scale.

**Task 2:** 4 numeric features stay as-is. The categorical feature with 5 unique values produces 5 − 1 = **4 one-hot columns** (with `drop_first=True` to avoid multicollinearity). Total: **4 + 4 = 8 columns**.

```python
# Task 3
loader_128 = DataLoader(train_dataset, batch_size=128, shuffle=True)
print(len(loader_128))   # ceil(len(train_dataset) / 128)
```